# CS4045 Deep Learning for Perception – Assignment 2
## Experimental Analysis of RNNs and LSTMs for Next-Word Prediction

**Name:** <MY NAME>, **Roll No:** <MY ROLL NO>

This single notebook contains every phase of the assignment:

| Phase | Content | Expected runtime, RTX 3080 (rough estimate) |
|---|---|---|
| 0 | Environment, paths, packages | < 1 min |
| 1 | Cleaning, sentence split, 80/10/10 split, vocabulary, `<s>`/`</s>`, sliding windows, BBC | 2–5 min (+ download on first run) |
| 2 | Vanishing-gradient experiment: RNN vs LSTM for k = 5…80 (10 runs × 5 epochs) | 10–25 min |
| 3 | 11 architecture runs with early stopping, model selection, best models saved to disk | 25–50 min |
| 4 | Chatbot with the best LSTM (greedy / temperature sampling) | < 1 min (+ your typing) |
| 5 | Sentence perplexity, length groups, regression, calibration | 1–3 min |
| 6 | BBC out-of-domain gap, frequency bands, specialised vs generalised fine-tuning | 5–10 min |
| 7 | Export tables/figures/JSON and README.txt | < 1 min |

A Colab T4 is roughly 2–3× slower than the 3080. Phases 2 and 3 cache their results, so a re-run skips finished experiments.

**How to run.** *Run all* from top to bottom. On Colab the Phase 4 chatbot waits for input: type `quit` to continue.
For a non-interactive run (university GPU server):

```
jupyter nbconvert --to notebook --execute A2.ipynb --output A2_executed.ipynb --ExecutePreprocessor.timeout=-1
```

In a non-interactive run the chatbot detects that no input box exists and closes itself. Optional switches are environment variables, so the notebook itself never needs editing for them: `A2_RUN_CHATBOT=0`, `A2_FORCE_RETRAIN=1`, `A2_BASE=/path/to/workdir`, `AUSTEN_DIR`, `SHAKESPEARE_DIR`, `BBC_DIR`, and `A2_QUICK_RUN=1` (a 2–5 minute smoke test on a small slice of the data; never use its numbers in the report).

## Phase 0 – Environment

**Cell 0.1 – Where am I running, and global switches.** The notebook detects whether it is running on Google Colab, Kaggle or a local/university machine; everything environment-specific depends on this one check, so no Colab-only code ever runs elsewhere. The switches below control the optional behaviour: whether the Phase 4 chatbot runs, whether cached Phase 2/3 results are ignored, and the quick smoke-test mode. Each can be set here or through an environment variable.

In [ ]:
import os, sys, subprocess, importlib.util

ON_COLAB  = 'google.colab' in sys.modules or 'COLAB_RELEASE_TAG' in os.environ
ON_KAGGLE = (not ON_COLAB) and os.path.isdir('/kaggle/working')
ENV = 'colab' if ON_COLAB else 'kaggle' if ON_KAGGLE else 'local'

RUN_CHATBOT   = os.environ.get('A2_RUN_CHATBOT', '1') != '0'    # Phase 4 interactive loop (default True)
FORCE_RETRAIN = os.environ.get('A2_FORCE_RETRAIN', '0') == '1'  # True = ignore cached Phase 2/3 results and retrain
QUICK_RUN     = os.environ.get('A2_QUICK_RUN', '0') == '1'      # tiny smoke test: NEVER use its numbers in the report
USE_DRIVE     = True                                            # Colab only: keep data/checkpoints/results on Google Drive

print(f'environment: {ENV} | python {sys.version.split()[0]}')
print(f'RUN_CHATBOT={RUN_CHATBOT} | FORCE_RETRAIN={FORCE_RETRAIN} | QUICK_RUN={QUICK_RUN}')
if QUICK_RUN:
    print('*** QUICK_RUN is ON: small data slice and very short training. Turn it off for the real results. ***')

**Cell 0.2 – Required packages.** Checks that every library the notebook imports is installed. On Colab and Kaggle any missing package is installed automatically (in practice they are all preinstalled). On a local or university machine nothing is installed behind your back: the cell stops with the exact `pip install` command to run. `kagglehub` is only needed if a dataset has to be downloaded, so it is optional.

In [ ]:
REQUIRED = {'torch': 'torch', 'numpy': 'numpy', 'pandas': 'pandas', 'matplotlib': 'matplotlib',
            'nltk': 'nltk', 'sklearn': 'scikit-learn', 'scipy': 'scipy'}
OPTIONAL = {'kagglehub': 'kagglehub'}

missing = [pkg for mod, pkg in REQUIRED.items() if importlib.util.find_spec(mod) is None]
missing_opt = [pkg for mod, pkg in OPTIONAL.items() if importlib.util.find_spec(mod) is None]
if ON_COLAB or ON_KAGGLE:
    if missing + missing_opt:
        print('installing:', missing + missing_opt)
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *(missing + missing_opt)])
        importlib.invalidate_caches()
elif missing:
    raise ImportError(f'Missing packages: {missing}. Install them first:\n'
                      f'  {sys.executable} -m pip install {" ".join(missing)}')
elif missing_opt:
    print(f'Note: {missing_opt} not installed. That is fine if the datasets are already on disk; '
          f'otherwise run: pip install {" ".join(missing_opt)}')
print('all required packages available')

**Cell 0.3 – Working folder (and Google Drive on Colab).** Everything the notebook writes (downloaded data, `phase1.pkl`, checkpoints, results, figures) goes under one folder, `BASE`. On Colab this is `/content/drive/MyDrive/DL_A2` as in my original notebook, so results survive a runtime reset; `drive.mount` only runs on Colab, and if it fails the notebook falls back to local Colab storage. On Kaggle it is `/kaggle/working/DL_A2`, and anywhere else `./DL_A2` next to the notebook. Set `BASE` by hand (or `A2_BASE`) to use another folder. The quick smoke test writes to a separate `_quick` folder so it can never overwrite real results.

In [ ]:
BASE = None          # <- override by hand, e.g. '/home/<you>/DL_A2'; None = choose automatically
if BASE is None and os.environ.get('A2_BASE'):
    BASE = os.environ['A2_BASE']
if BASE is None:
    if ON_COLAB:
        BASE = '/content/DL_A2'
        if USE_DRIVE:
            try:
                from google.colab import drive          # Colab-only import, guarded
                drive.mount('/content/drive')
                BASE = '/content/drive/MyDrive/DL_A2'
            except Exception as e:
                print(f'Google Drive not mounted ({e}); using {BASE} (deleted when the runtime ends).')
    elif ON_KAGGLE:
        BASE = '/kaggle/working/DL_A2'
    else:
        BASE = os.path.abspath('DL_A2')
if QUICK_RUN:
    BASE = BASE.rstrip('/') + '_quick'
BASE = os.path.abspath(BASE)
os.makedirs(BASE, exist_ok=True)
print('BASE =', BASE)

## Phase 1 – Sequential Data Engineering

**Cell 1.1 – Locating the three datasets.** Each dataset has one override variable (`AUSTEN_DIR`, `SHAKESPEARE_DIR`, `BBC_DIR`). Leave them as `None` and the folder is found automatically, in this order:
1. `BASE/data/<name>` (where my original notebook copied the downloads);
2. Kaggle `/kaggle/input/...`, Colab `/content/...`, local `./data/...` and `~/data/...`, matched by the Kaggle dataset name;
3. a fresh `kagglehub` download, copied into `BASE/data/<name>`.

A folder is only accepted if it really contains the expected files (Austen `.txt` files, Shakespeare `.csv`/`.txt`, BBC category folders), so a wrong folder fails loudly here instead of producing an empty corpus later. The listing at the end shows what was found.

In [ ]:
import shutil, glob

DATA = f'{BASE}/data'
OUT  = f'{BASE}/processed'
os.makedirs(DATA, exist_ok=True)
os.makedirs(OUT, exist_ok=True)

# ---- one override per dataset: the folder that contains the dataset files (None = auto-detect) ----
AUSTEN_DIR      = None or os.environ.get('AUSTEN_DIR')        # Kaggle: joshmcadams/jane-austin-and-charles-dickens
SHAKESPEARE_DIR = None or os.environ.get('SHAKESPEARE_DIR')   # Kaggle: thedevastator/the-bards-best-a-character-modeling-dataset
BBC_DIR         = None or os.environ.get('BBC_DIR')           # Kaggle: shivamkushwaha/bbc-full-text-document-classification

datasets = {
    'novels':      'joshmcadams/jane-austin-and-charles-dickens',
    'shakespeare': 'thedevastator/the-bards-best-a-character-modeling-dataset',
    'bbc':         'shivamkushwaha/bbc-full-text-document-classification',
}
CHECKS = {   # what a valid dataset folder must contain
    'novels':      lambda d: any('aust' in os.path.relpath(f, d).lower()
                                 for f in glob.glob(f'{d}/**/*.txt', recursive=True)),
    'shakespeare': lambda d: bool(glob.glob(f'{d}/**/*.csv', recursive=True) + glob.glob(f'{d}/**/*.txt', recursive=True)),
    'bbc':         lambda d: bool(glob.glob(f'{d}/**/business/*.txt', recursive=True)),
}

def locate_dataset(name, slug, override=None):
    if override:
        override = os.path.abspath(os.path.expanduser(override))
        assert os.path.isdir(override) and CHECKS[name](override), \
            f'{name}: the folder you set ({override}) does not contain the expected files'
        return override
    leaf = slug.split('/')[1]
    cands = [f'{DATA}/{name}']
    for root in ['/kaggle/input', '/content', '/content/data', './data', os.path.expanduser('~/data')]:
        for pat in [f'{root}/{name}', f'{root}/{leaf}', f'{root}/*/{leaf}', f'{root}/*/*/{leaf}']:
            cands += sorted(glob.glob(pat))
    for c in cands:
        if os.path.isdir(c) and CHECKS[name](c):
            return os.path.abspath(c)
    # not on disk: download with kagglehub (my original approach) and copy into BASE/data/<name>
    try:
        import kagglehub
    except ImportError as e:
        raise FileNotFoundError(f'{name}: dataset not found and kagglehub is not installed. Download '
                                f'kaggle.com/datasets/{slug}, unzip it and set the *_DIR variable above.') from e
    dst = f'{DATA}/{name}'
    shutil.copytree(kagglehub.dataset_download(slug), dst, dirs_exist_ok=True)
    assert CHECKS[name](dst), f'{name}: download finished but the expected files are missing in {dst}'
    return dst

AUSTEN_DIR      = locate_dataset('novels',      datasets['novels'],      AUSTEN_DIR)
SHAKESPEARE_DIR = locate_dataset('shakespeare', datasets['shakespeare'], SHAKESPEARE_DIR)
BBC_DIR         = locate_dataset('bbc',         datasets['bbc'],         BBC_DIR)

# Look at what was found
for label, d in [('AUSTEN_DIR', AUSTEN_DIR), ('SHAKESPEARE_DIR', SHAKESPEARE_DIR), ('BBC_DIR', BBC_DIR)]:
    print(f'{label} = {d}')
    for root, dirs, files in os.walk(d):
        if root.replace(d, '').count(os.sep) <= 3:
            print('   ', root, '|', len(files), 'files |', files[:4])

**Cell 1.2 – Imports, seed, tokenizer.** Fixes the random seed (42) for Python, NumPy and PyTorch. The tokenizer lowercases text and keeps words and contractions (`don't`, `o'er`); punctuation and numbers are dropped. Curly quotes are normalised first so `don’t` and `don't` become the same token. Sentences are split with NLTK's Punkt model. Its data files are downloaded only if they are missing; if the download is impossible (for example behind a proxy) the cell stops with instructions rather than failing later inside `sent_tokenize`.

In [ ]:
import re, glob, random, pickle, hashlib
from collections import Counter
import numpy as np, pandas as pd, torch
import nltk
from nltk.tokenize import sent_tokenize

for _res in ['punkt', 'punkt_tab']:
    try:
        nltk.data.find(f'tokenizers/{_res}')
    except LookupError:
        try:
            nltk.download(_res, quiet=True)
        except Exception as e:
            print(f'nltk.download({_res!r}) failed: {e}')
try:
    sent_tokenize('A first sentence. A second one.')
except LookupError as e:
    raise RuntimeError('NLTK sentence-tokenizer data is missing and could not be downloaded. Run once in a terminal:\n'
                       f'  {sys.executable} -m nltk.downloader punkt punkt_tab') from e

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

WORD_RE = re.compile(r"[a-z]+(?:'[a-z]+)*")
def tokenize(sentence):
    """Lowercase; keep words and contractions; drop punctuation and numbers."""
    return WORD_RE.findall(sentence.lower())

def normalize_quotes(t):
    return (t.replace('\r\n', '\n').replace('’', "'").replace('‘', "'")
             .replace('“', '"').replace('”', '"'))

def to_sentences(paragraphs):
    out = []
    for p in paragraphs:
        for s in sent_tokenize(p):
            toks = tokenize(s)
            if toks:
                out.append(toks)
    return out

**Cell 1.3 – Cleaning the Jane Austen text and removing the Gutenberg noise.** Only the text between the `*** START OF … PROJECT GUTENBERG ***` and `*** END OF … ***` markers is kept; this also handles several books in one file. Inside that span the cell removes `[Illustration]` and `[Footnote …]` blocks and `_italics_` markers. It also drops lines mentioning Gutenberg, e-books, transcribers, URLs or copyright, short `CHAPTER IV` / `VOLUME II` headings, and short all-caps lines such as titles, `THE END` and `* * *`. Lines inside a paragraph are joined so sentences broken across lines are reassembled. Only files with “aust” in their path are used, which matches both “Austen” and the dataset's “Austin” spelling, so the Dickens novels in the same download are ignored.

In [ ]:
START_RE = re.compile(r'\*\*\*\s*START OF (?:THE |THIS )?PROJECT GUTENBERG[^\n]*', re.I)
END_RE   = re.compile(r'\*\*\*\s*END OF (?:THE |THIS )?PROJECT GUTENBERG[^\n]*', re.I)
NOISE_LINE = re.compile(
    r'gutenberg|e-?book|produced by|transcriber|http|www\.|copyright|'
    r'^(title|author|release date|posting date|language|character set|last updated|edition)\s*:', re.I)
HEADING = re.compile(r'^\s*(chapter|volume)\s+[ivxlcdm\d]+\.?(\s.*)?$', re.I)

def strip_gutenberg(text):
    """Keep only text between START/END markers (handles several books in one file)."""
    starts = [m.end() for m in START_RE.finditer(text)]
    ends   = [m.start() for m in END_RE.finditer(text)]
    if not starts:
        return text
    parts = []
    for s in starts:
        e = min([x for x in ends if x > s], default=len(text))
        parts.append(text[s:e])
    return '\n\n'.join(parts)

def clean_austen(text):
    text = strip_gutenberg(normalize_quotes(text))
    text = re.sub(r'\[[^\]]{0,500}\]', ' ', text)   # [Illustration], [Footnote ...]
    text = text.replace('_', '')                     # _italics_ markup
    paras = []
    for block in re.split(r'\n\s*\n', text):
        lines = [l.strip() for l in block.split('\n') if l.strip()]
        lines = [l for l in lines if not NOISE_LINE.search(l)]
        lines = [l for l in lines if not (len(l) < 60 and HEADING.match(l))]   # CHAPTER IV
        lines = [l for l in lines if not (len(l) < 60 and l.upper() == l)]     # ALL-CAPS titles, THE END, * * *
        if lines:
            paras.append(' '.join(lines))
    return paras

# 'aust' matches both "austen" and the dataset's "austin" spelling
austen_files = sorted(f for f in glob.glob(f'{AUSTEN_DIR}/**/*.txt', recursive=True)
                      if 'aust' in os.path.relpath(f, AUSTEN_DIR).lower())
assert austen_files, 'No Austen .txt found: check the listing from Cell 1.1 and adjust the pattern'
print('Austen files:', [os.path.basename(f) for f in austen_files])

austen_paras = []
for f in austen_files:
    austen_paras += clean_austen(open(f, encoding='utf-8', errors='ignore').read())
austen_sents = to_sentences(austen_paras)
print(f'Austen: {len(austen_sents):,} sentences, {sum(map(len, austen_sents)):,} words')
print('Sample:', ' '.join(austen_sents[100]))

**Cell 1.4 – Loading and cleaning Tiny Shakespeare.** Speaker labels such as `First Citizen:` are removed, since they are not part of the language being modelled. The dataset's own train/validation/test files are merged, because the assignment requires a different split (Cell 1.5). The cell warns if the download contains both `.csv` and `.txt` copies of the plays, which would put the same text in the corpus twice.

In [ ]:
SPEAKER = re.compile(r"^[A-Za-z][A-Za-z' .-]{0,40}:$")

def read_any(path):
    if path.endswith('.csv'):
        df = pd.read_csv(path)
        col = 'text' if 'text' in df.columns else df.select_dtypes('object').columns[0]
        return '\n'.join(df[col].dropna().astype(str))
    return open(path, encoding='utf-8', errors='ignore').read()

def clean_shakespeare(text):
    paras = []
    for block in re.split(r'\n\s*\n', normalize_quotes(text)):
        lines = [l.strip() for l in block.split('\n') if l.strip()]
        lines = [l for l in lines if not SPEAKER.match(l)]
        if lines:
            paras.append(' '.join(lines))
    return paras

shk_files = sorted(glob.glob(f'{SHAKESPEARE_DIR}/**/*.csv', recursive=True) +
                   glob.glob(f'{SHAKESPEARE_DIR}/**/*.txt', recursive=True))
assert shk_files, 'No Shakespeare files found'
print('Shakespeare files:', [os.path.basename(f) for f in shk_files])
if any(f.endswith('.csv') for f in shk_files) and any(f.endswith('.txt') for f in shk_files):
    print('WARNING: both .csv and .txt files found - check they are not two copies of the same plays.')

shk_sents = to_sentences(clean_shakespeare('\n\n'.join(read_any(f) for f in shk_files)))
print(f'Shakespeare: {len(shk_sents):,} sentences, {sum(map(len, shk_sents)):,} words')
print('Sample:', ' '.join(shk_sents[50]))

**Cell 1.5 – 80/10/10 split by blocks of sentences.** Each corpus is cut into blocks of 20 consecutive sentences. The blocks are shuffled with the fixed seed and divided 80/10/10, and the blocks inside each split are put back in their original order so the text stays coherent. Splitting whole blocks rather than individual windows keeps nearly identical overlapping windows from landing in both train and test. Austen and Shakespeare are split separately and then combined, so each split contains both styles in the same proportion. This happens before the vocabulary is built.

In [ ]:
BLOCK = 20  # consecutive sentences per block

if QUICK_RUN:                                   # smoke test only: a small slice of each corpus
    austen_sents, shk_sents = austen_sents[:4000], shk_sents[:2000]

def block_split(sents, block=BLOCK, ratios=(0.8, 0.1, 0.1), seed=SEED):
    blocks = [sents[i:i+block] for i in range(0, len(sents), block)]
    idx = np.random.RandomState(seed).permutation(len(blocks))
    n_tr, n_va = int(ratios[0]*len(blocks)), int(ratios[1]*len(blocks))
    pick = lambda ids: [s for b in sorted(ids) for s in blocks[b]]
    return pick(idx[:n_tr]), pick(idx[n_tr:n_tr+n_va]), pick(idx[n_tr+n_va:])

a_tr, a_va, a_te = block_split(austen_sents)
s_tr, s_va, s_te = block_split(shk_sents)
train_sents, val_sents, test_sents = a_tr + s_tr, a_va + s_va, a_te + s_te

for name, s in [('train', train_sents), ('val', val_sents), ('test', test_sents)]:
    print(f'{name:5s}: {len(s):7,} sentences, {sum(map(len, s)):9,} words')

**Cell 1.6 – Vocabulary from the training split only, plus `<UNK>` rates.** Words seen fewer than `MIN_COUNT = 3` times in the training split are mapped to `<UNK>`. The vocabulary is ordered by frequency, after four special tokens: `<pad>` = 0, `<UNK>` = 1, `<s>` = 2, `</s>` = 3. The `<UNK>` rate is the share of word tokens in each split that fall outside the vocabulary. Validation and test rates are higher than train because they also contain words never seen in training.

In [ ]:
MIN_COUNT = 3   # state this in your report
PAD, UNK, BOS, EOS = '<pad>', '<UNK>', '<s>', '</s>'

train_counts = Counter(w for s in train_sents for w in s)
itos = [PAD, UNK, BOS, EOS] + [w for w, c in train_counts.most_common() if c >= MIN_COUNT]
stoi = {w: i for i, w in enumerate(itos)}
PAD_ID, UNK_ID, BOS_ID, EOS_ID = 0, 1, 2, 3

def unk_rate(sents):
    """Share of word tokens mapped to <UNK> (special tokens excluded)."""
    total = sum(len(s) for s in sents)
    unk = sum(1 for s in sents for w in s if w not in stoi)
    return unk / total

print(f'Min count: {MIN_COUNT}')
print(f'Vocabulary size (incl. 4 special tokens): {len(itos):,}')
for name, s in [('train', train_sents), ('val', val_sents), ('test', test_sents)]:
    print(f'<UNK> rate {name:5s}: {100*unk_rate(s):.2f}%')

**Cell 1.7 – Adding `<s>` and `</s>`.** Each split becomes one continuous stream of word ids, with `<s>` before and `</s>` after every sentence. `spans` records where each sentence starts and ends in the stream.

In [ ]:
def encode(sents):
    ids, spans = [], []
    for s in sents:
        start = len(ids)
        ids.append(BOS_ID)
        ids.extend(stoi.get(w, UNK_ID) for w in s)
        ids.append(EOS_ID)
        spans.append((start, len(ids)))      # stream[start] is <s>, stream[end-1] is </s>
    return np.array(ids, dtype=np.int32), spans

train_ids, train_spans = encode(train_sents)
val_ids,   val_spans   = encode(val_sents)
test_ids,  test_spans  = encode(test_sents)
print('First 30 train tokens:', ' '.join(itos[i] for i in train_ids[:30]))

**Cell 1.8 – Sliding-window tokenizer.** `sliding_windows` is the textbook version, shown on the assignment's example: with k = 3, “it is a truth …” gives input “it is a” and target “truth”.

`NextWordDataset` does the same thing efficiently on the id stream. It stores only the target positions and slices the k previous tokens when a batch is requested. `<s>` is never a target, as the assignment requires; `</s>` is. Because the stream is continuous, a window can reach back into previous sentences. `first_target` lets Phase 2 use the same targets for every k. `batches()` builds whole batches with one indexing operation, which is much faster than a `DataLoader` calling `__getitem__` per window.

In [ ]:
def sliding_windows(tokens, k):
    """Return (input of k tokens, next-token target) pairs."""
    return [(tokens[i-k:i], tokens[i]) for i in range(k, len(tokens))]

for x, y in sliding_windows('it is a truth universally acknowledged'.split(), k=3):
    print(x, '->', y)

class NextWordDataset(torch.utils.data.Dataset):
    def __init__(self, stream, k, first_target=None):
        """first_target: earliest target position. For Phase 2, set it to 80 for every k
        so all window sizes are evaluated on exactly the same target words."""
        self.stream = torch.from_numpy(stream.astype(np.int64))
        self.k = k
        first = k if first_target is None else first_target
        assert first >= k
        pos = np.arange(first, len(stream))
        pos = pos[stream[pos] != BOS_ID]          # <s> is never predicted
        self.pos = torch.from_numpy(pos)
        self.offsets = torch.arange(-k, 0)

    def __len__(self):
        return len(self.pos)

    def __getitem__(self, i):
        t = self.pos[i]
        return self.stream[t - self.k:t], self.stream[t]

    def batches(self, batch_size, shuffle=False, seed=SEED):
        """Fast vectorised batching (much quicker than a DataLoader with __getitem__)."""
        order = (torch.randperm(len(self.pos), generator=torch.Generator().manual_seed(seed))
                 if shuffle else torch.arange(len(self.pos)))
        for b in range(0, len(order), batch_size):
            t = self.pos[order[b:b + batch_size]]
            yield self.stream[t[:, None] + self.offsets], self.stream[t]

K = 20
train_ds = NextWordDataset(train_ids, K)
val_ds   = NextWordDataset(val_ids, K)
test_ds  = NextWordDataset(test_ids, K)
print(f'k={K}: train {len(train_ds):,} | val {len(val_ds):,} | test {len(test_ds):,} windows')

x, y = train_ds[0]
print('INPUT :', ' '.join(itos[i] for i in x.tolist()))
print('TARGET:', itos[y.item()])

**Cell 1.9 – BBC News, deduplicated and encoded with the training vocabulary.** The download contains two copies of every article. Duplicates are removed by (category, file name), which should leave exactly 2,225 articles. The title line is dropped and only the body text is kept. BBC text goes through exactly the same sentence splitting and tokenizer as the primary corpus, and is encoded with the frozen training vocabulary, so unseen words become `<UNK>`. The cell also counts distinct article bodies: the original BBC set contains a few exact repeats under different file names. They are kept here and removed before the Phase 6 adaptation split (Cell 6.1). Each article (category, file, body) is stored in `bbc_articles` so Phase 6 reuses this exact preprocessing.

In [ ]:
CATS = {'business', 'entertainment', 'politics', 'sport', 'tech'}
unique = {}
for f in sorted(glob.glob(f'{BBC_DIR}/**/*.txt', recursive=True)):
    cat = os.path.basename(os.path.dirname(f))
    if cat in CATS:
        unique.setdefault((cat, os.path.basename(f)), f)
print('BBC files after removing duplicates:', len(unique))   # expect 2225
if QUICK_RUN:                                   # smoke test only: first 40 articles per category
    unique = {key: f for key, f in unique.items() if int(re.sub(r'\D', '', key[1]) or 0) <= 40}

def bbc_body_paragraphs(body):
    return [' '.join(p.split()) for p in re.split(r'\n\s*\n', body) if p.strip()]

bbc_paras, hashes, bbc_articles = [], set(), []
for (cat, fname), f in unique.items():
    text = normalize_quotes(open(f, encoding='latin-1').read())
    body = '\n'.join(text.split('\n')[1:])                   # drop title line
    h = hashlib.md5(body.strip().encode()).hexdigest()
    hashes.add(h)
    bbc_paras += bbc_body_paragraphs(body)
    bbc_articles.append(dict(category=cat, file=fname, body=body, md5=h))
print('Distinct article bodies by content:', len(hashes))

bbc_sents = to_sentences(bbc_paras)
bbc_ids, bbc_spans = encode(bbc_sents)
print(f'BBC: {len(bbc_sents):,} sentences, {sum(map(len, bbc_sents)):,} words')
print(f'<UNK> rate BBC: {100*unk_rate(bbc_sents):.2f}%')

**Cell 1.10 – Summary table and saving.** This is the Phase 1 table for the report: sentences, words and `<UNK>` rate for each split and for BBC. Everything later phases need is saved to `BASE/processed/phase1.pkl`. A short fingerprint of the vocabulary and splits is saved with it; Phases 2 and 3 use it to detect cached results that were trained on different data.

In [ ]:
summary = pd.DataFrame([
    {'split': n, 'sentences': len(s), 'words': sum(map(len, s)),
     'UNK rate (%)': round(100*unk_rate(s), 2)}
    for n, s in [('train', train_sents), ('val', val_sents),
                 ('test', test_sents), ('BBC (OOD)', bbc_sents)]
])
print(f'Vocab size = {len(itos):,} | min count = {MIN_COUNT} | seed = {SEED} | block = {BLOCK}')
display(summary)

DATA_FP = hashlib.md5(('\n'.join(itos) + '|' + '|'.join(
    hashlib.md5(a.tobytes()).hexdigest() for a in (train_ids, val_ids, test_ids))).encode()).hexdigest()[:12]
with open(f'{OUT}/phase1.pkl', 'wb') as f:
    pickle.dump({
        'itos': itos, 'stoi': stoi, 'train_counts': train_counts,
        'min_count': MIN_COUNT, 'seed': SEED, 'data_fingerprint': DATA_FP,
        'train_ids': train_ids, 'val_ids': val_ids, 'test_ids': test_ids, 'bbc_ids': bbc_ids,
        'train_spans': train_spans, 'val_spans': val_spans,
        'test_spans': test_spans, 'bbc_spans': bbc_spans,
    }, f)
print('Saved to', f'{OUT}/phase1.pkl', '| data fingerprint', DATA_FP)

## Shared setup (used by Phases 2–6)

**Cell S.1 – Reload Phase 1, folders, device.** This cell reloads everything Phase 1 saved, so it also works after a runtime restart (re-run the Phase 0 cells, Cell 1.2 and Cell 1.8 first). It creates `checkpoints/` and `results/` under `BASE`. The device is chosen automatically: CUDA if available, otherwise CPU. `USE_AMP` turns on mixed precision (float16) on the GPU only, which roughly doubles speed on a T4 and speeds up the 3080 too.

Phases 2 and 3 skip experiments whose results are already saved. To make that safe, the data fingerprint from Cell 1.10 is compared with the one stored next to the results. If they differ, the cached results came from different data or a different vocabulary, so they are ignored and the models are retrained. `make_scaler` creates the mixed-precision gradient scaler on both new and older PyTorch versions.

In [ ]:
import os, json, math, time, random, pickle
import numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt

OUT  = f'{BASE}/processed'
CKPT = f'{BASE}/checkpoints'; os.makedirs(CKPT, exist_ok=True)
RES  = f'{BASE}/results';     os.makedirs(RES, exist_ok=True)

with open(f'{OUT}/phase1.pkl', 'rb') as f:
    P1 = pickle.load(f)
itos, stoi = P1['itos'], P1['stoi']
train_ids, val_ids, test_ids = P1['train_ids'], P1['val_ids'], P1['test_ids']
V = len(itos)
PAD_ID, UNK_ID, BOS_ID, EOS_ID = 0, 1, 2, 3
SEED = P1['seed']
DATA_FP = P1['data_fingerprint']

device  = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_AMP = device.type == 'cuda'          # mixed precision only on GPU

def set_seed(s=SEED):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

def make_scaler():
    try:
        return torch.amp.GradScaler('cuda', enabled=USE_AMP)
    except (AttributeError, TypeError):                # PyTorch < 2.3
        return torch.cuda.amp.GradScaler(enabled=USE_AMP)

# ---- cached results are only reused if they were produced from the same data ----
fp_file = f'{RES}/data_fingerprint.txt'
old_fp = open(fp_file).read().strip() if os.path.exists(fp_file) else None
CACHE_VALID = not FORCE_RETRAIN and (old_fp is None or old_fp == DATA_FP)
if old_fp is not None and old_fp != DATA_FP:
    print(f'NOTE: cached results were made from different data ({old_fp} != {DATA_FP}); they will be recomputed.')
with open(fp_file, 'w') as f:
    f.write(DATA_FP)

assert 'NextWordDataset' in globals() and 'tokenize' in globals(), \
    'Re-run the Phase 1 tokenize cell and NextWordDataset cell first'
if device.type == 'cuda':
    print('Device:', device, torch.cuda.get_device_name(0), f'| torch {torch.__version__} | CUDA {torch.version.cuda}')
else:
    print('Device: cpu  (WARNING: no GPU visible - training will be very slow)')
print(f'Vocab {V:,} | train tokens {len(train_ids):,} | val {len(val_ids):,} | test {len(test_ids):,}')
print(f'checkpoints -> {CKPT}\nresults     -> {RES}\nreuse cached Phase 2/3 results: {CACHE_VALID}')

**Cell S.2 – The model.** One class covers both architectures: an embedding layer, a recurrent layer (`nn.LSTM`, or `nn.RNN`, which is the Elman RNN with tanh), and a linear layer that turns the last hidden state into a score for every vocabulary word. Only the last timestep predicts the next word. Dropout is applied to the input embeddings and the final hidden state, and also between stacked layers when there is more than one. `forward_emb` takes embeddings directly so Phase 2 can measure gradients with respect to each input word.

In [ ]:
class NextWordModel(nn.Module):
    def __init__(self, vocab_size, cell='lstm', emb_dim=256, hidden=256, layers=1, dropout=0.0):
        super().__init__()
        self.config = dict(cell=cell, emb_dim=emb_dim, hidden=hidden, layers=layers, dropout=dropout)
        self.embed = nn.Embedding(vocab_size, emb_dim, padding_idx=PAD_ID)
        RNN = nn.LSTM if cell == 'lstm' else nn.RNN          # nn.RNN = Elman RNN with tanh
        self.rnn = RNN(emb_dim, hidden, num_layers=layers, batch_first=True,
                       dropout=dropout if layers > 1 else 0.0)  # between stacked layers
        self.drop = nn.Dropout(dropout)                          # on input embeddings and final state
        self.head = nn.Linear(hidden, vocab_size)                # hidden -> score for every word

    def forward_emb(self, e):                 # e: (batch, k, emb_dim)
        out, _ = self.rnn(self.drop(e))       # out: (batch, k, hidden)
        return self.head(self.drop(out[:, -1]))  # only the last timestep predicts the next word

    def forward(self, x):                     # x: (batch, k) word ids
        return self.forward_emb(self.embed(x))

def count_params(m):
    return sum(p.numel() for p in m.parameters())

**Cell S.3 – Training and evaluation.** `evaluate` computes perplexity = exp(mean cross-entropy) over every window in a dataset, plus top-1 accuracy, which the report asks for. The loss is summed in float32, so the result is exact.

`train_epoch` runs one pass over the shuffled training windows. The shuffle seed changes each epoch but stays reproducible. Gradients are clipped to norm 1.0, which stops the Simple RNN from blowing up through exploding gradients. Clipping does nothing against vanishing gradients, so the Phase 2 comparison stays fair. The loss is accumulated on the GPU, so there is no `.item()` sync on every step.

`fit` handles both phases. With `patience=None` (Phase 2) it trains for a fixed number of epochs at a constant learning rate. With a patience value (Phase 3) it halves the learning rate the first time validation perplexity fails to improve, stops after `patience` epochs in a row without improvement, and saves the best checkpoint. Either way it reloads the best-validation weights before returning.

In [ ]:
@torch.no_grad()
def evaluate(model, ds, batch_size=1024):
    model.eval()
    tot_loss, correct, n = 0.0, 0, 0
    for x, y in ds.batches(batch_size):
        x, y = x.to(device), y.to(device)
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=USE_AMP):
            logits = model(x)
        tot_loss += F.cross_entropy(logits.float(), y, reduction='sum').item()
        correct  += (logits.argmax(1) == y).sum().item()
        n        += y.numel()
    return math.exp(tot_loss / n), correct / n          # (perplexity, accuracy)

def train_epoch(model, ds, opt, scaler, batch_size, seed, clip=1.0):
    model.train()
    total, count = torch.zeros((), device=device), 0
    for x, y in ds.batches(batch_size, shuffle=True, seed=seed):
        x, y = x.to(device), y.to(device)
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=USE_AMP):
            logits = model(x)
        loss = F.cross_entropy(logits.float(), y)
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()                       # scaled to avoid fp16 underflow
        scaler.unscale_(opt)
        nn.utils.clip_grad_norm_(model.parameters(), clip)  # same clipping for RNN and LSTM
        scaler.step(opt); scaler.update()
        total += loss.detach() * y.numel(); count += y.numel()
    return (total / count).item()

def fit(model, train_ds, val_ds, epochs, lr=1e-3, batch_size=512, patience=None, ckpt=None):
    opt    = torch.optim.Adam(model.parameters(), lr=lr)
    sched  = (torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode='min', factor=0.5, patience=0)
              if patience else None)
    scaler = make_scaler()
    hist, best, best_state, bad = [], float('inf'), None, 0
    for ep in range(1, epochs + 1):
        t0 = time.time()
        tr_loss = train_epoch(model, train_ds, opt, scaler, batch_size, seed=SEED + ep)
        val_ppl, val_acc = evaluate(model, val_ds)
        secs = time.time() - t0
        hist.append(dict(epoch=ep, train_ppl=math.exp(tr_loss), val_ppl=val_ppl,
                         val_acc=val_acc, lr=opt.param_groups[0]['lr'], secs=secs))
        print(f'  ep {ep:2d} | train ppl {math.exp(tr_loss):8.2f} | val ppl {val_ppl:8.2f} '
              f'| val acc {100*val_acc:5.2f}% | {secs:5.0f}s')
        if val_ppl < best:
            best, bad = val_ppl, 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            if ckpt: torch.save(best_state, ckpt)
        else:
            bad += 1
        if sched: sched.step(val_ppl)
        if patience and bad >= patience:
            print(f'  early stop: no improvement for {patience} epochs'); break
    model.load_state_dict(best_state)          # return the best-validation weights
    return hist, best

## Phase 2 – The Vanishing Gradient Experiment

**Cell 2.1 – Train an RNN and an LSTM for k = 5, 10, 20, 40, 80.** The key detail is `first_target=80` on both the train and validation datasets. Every k then predicts exactly the same target words, and only the amount of context changes: with k = 5 the model sees the 5 words before each target, with k = 80 the 80 words before the same target. Because the token stream runs continuously, a long window spans earlier sentences, including their `</s>` and `<s>` markers. That is what lets a long k test long-range memory.

Both models get identical settings, which go in the report: embedding 256, hidden 256, 1 layer, no dropout, Adam with lr 1e-3, batch 512, 5 epochs, gradient clipping at 1.0, seed 42. Weight initialisation is seeded the same way for each run. The k = 80 models are saved for Cell 2.3. Results are written to `phase2.json` after every run. Finished runs are skipped on a re-run, but only if their data fingerprint matches and the k = 80 checkpoint exists.

In [ ]:
P2_KS     = [5, 10, 20, 40, 80]
P2_CFG    = dict(emb_dim=256, hidden=256, layers=1, dropout=0.0)
P2_EPOCHS, P2_LR, P2_BS = 5, 1e-3, 512
FIRST     = max(P2_KS)                      # same target positions for every k
if QUICK_RUN:
    P2_EPOCHS = 1

p2_path = f'{RES}/phase2.json'
p2 = json.load(open(p2_path)) if (os.path.exists(p2_path) and CACHE_VALID) else {}

for k in P2_KS:
    tr = NextWordDataset(train_ids, k, first_target=FIRST)
    va = NextWordDataset(val_ids,   k, first_target=FIRST)
    for cell in ['rnn', 'lstm']:
        key = f'{cell}_k{k}'
        ckpt_ok = k != FIRST or os.path.exists(f'{CKPT}/p2_{key}.pt')
        if key in p2 and ckpt_ok:
            print(f'{key}: already done (val ppl {p2[key]["best_val_ppl"]:.2f}), skipping'); continue
        set_seed(SEED)
        model = NextWordModel(V, cell, **P2_CFG).to(device)
        print(f'--- {key}: {count_params(model):,} params | {len(tr):,} train / {len(va):,} val targets')
        hist, best = fit(model, tr, va, epochs=P2_EPOCHS, lr=P2_LR, batch_size=P2_BS)
        p2[key] = dict(cell=cell, k=k, best_val_ppl=best,
                       final_val_ppl=hist[-1]['val_ppl'], history=hist)
        if k == FIRST:
            torch.save(model.state_dict(), f'{CKPT}/p2_{key}.pt')
        json.dump(p2, open(p2_path, 'w'), indent=1)
        del model
        if device.type == 'cuda': torch.cuda.empty_cache()

**Cell 2.2 – Results table and the perplexity-vs-k plot.** This builds a table of best and final validation perplexity for each model and k, then plots perplexity against k on a log-scale x-axis, since the k values double. The figure is saved to `results/` for the report. The usual pattern is that both models improve from k = 5 to around k = 20. Beyond that, the LSTM stays flat or keeps improving slightly, while the Simple RNN stops improving or gets worse: it cannot make use of the extra distant context, and the longer chain of timesteps makes it harder to train.

In [ ]:
df2 = pd.DataFrame([{'model': v['cell'].upper(), 'k': v['k'],
                     'best val ppl': round(v['best_val_ppl'], 2),
                     'final val ppl': round(v['final_val_ppl'], 2)} for v in p2.values()])
display(df2.pivot(index='k', columns='model', values='best val ppl'))

plt.figure(figsize=(6.5, 4.2))
for cell, mk in [('RNN', 'o-'), ('LSTM', 's-')]:
    d = df2[df2.model == cell].sort_values('k')
    plt.plot(d.k, d['best val ppl'], mk, label=f'Simple {cell}' if cell == 'RNN' else cell)
plt.xscale('log'); plt.xticks(P2_KS, P2_KS)
plt.xlabel('window size k (words of context)'); plt.ylabel('validation perplexity')
plt.title('Phase 2: validation perplexity vs. window size (same targets)')
plt.grid(alpha=.3); plt.legend(); plt.tight_layout()
plt.savefig(f'{RES}/phase2_ppl_vs_k.png', dpi=200); plt.show()

**Cell 2.3 – Measuring the vanishing gradient directly.** The perplexity plot shows the vanishing gradient indirectly; this cell measures it. It takes 512 validation windows with k = 80, backpropagates the loss, and records how large the gradient is at the input embedding of each earlier word. Distance 1 is the word just before the target and distance 80 is the furthest one. For the Simple RNN the curve typically falls by several orders of magnitude, which means far-back words have almost no effect on learning. For the LSTM it falls much more slowly.

The model is switched to `train()` mode only because cuDNN refuses to run RNN backward in eval mode. Dropout is 0 in these models, so the computation is the same either way. If the RNN's gradient at distance 80 is so small that it underflows to exactly 0 in float32, the ratio is printed as “inf” and the log-scale plot simply stops where the gradient reached 0.

In [ ]:
def grad_by_distance(model, ds, n=512):
    model.train()                                      # cuDNN needs train mode for backward (dropout=0 anyway)
    x, y = next(ds.batches(n, shuffle=True))
    x, y = x.to(device), y.to(device)
    e = model.embed(x).detach().requires_grad_(True)   # track gradient w.r.t. each input word
    F.cross_entropy(model.forward_emb(e).float(), y).backward()
    g = e.grad.norm(dim=2).mean(0)                     # avg gradient size at each of the k positions
    model.zero_grad(set_to_none=True)
    return g.flip(0).cpu().numpy()                     # index 0 = 1 word before the target

va80 = NextWordDataset(val_ids, 80, first_target=80)
plt.figure(figsize=(6.5, 4.2))
for cell in ['rnn', 'lstm']:
    m = NextWordModel(V, cell, **P2_CFG).to(device)
    m.load_state_dict(torch.load(f'{CKPT}/p2_{cell}_k80.pt', map_location=device))
    g = grad_by_distance(m, va80)
    plt.semilogy(np.arange(1, 81), g, label='Simple RNN' if cell == 'rnn' else 'LSTM')
    ratio = f'{g[0]/g[-1]:.1e}' if g[-1] > 0 else 'inf (gradient underflowed to 0: fully vanished)'
    print(f'{cell}: grad at distance 1 = {g[0]:.2e}, at distance 80 = {g[-1]:.2e}, ratio = {ratio}')
plt.xlabel('distance from target (words back)'); plt.ylabel('mean ||dLoss / d embedding|| (log)')
plt.title('Gradient reaching each input position (k = 80)')
plt.grid(alpha=.3); plt.legend(); plt.tight_layout()
plt.savefig(f'{RES}/phase2_grad_vs_distance.png', dpi=200); plt.show()

## Phase 3 – Hyperparameter and Architecture Research

**Cell 3.1 – Datasets for k = 20 and the experiment list.** From here on k = 20, as the assignment requires. Every window starts at position 20 (the default), because there is no cross-k comparison any more.

The 11 experiments vary one factor at a time around a baseline (1 layer, 256 hidden units), so each comparison in the report isolates the effect of a single factor. The 3 RNN runs (minimum 2) cover depth, width and dropout. The LSTM runs move along three axes: width 128 → 256 → 512; dropout 0 → 0.2 → 0.5 at a fixed size; depth 1 → 2 → 3 layers. Larger models are paired with higher dropout because they overfit faster on a corpus of about 1M words. Every value stays within the allowed ranges: layers 1–3, hidden {128, 256, 512}, dropout {0, 0.2, 0.5}. If time is tight, deleting the last two lines still leaves 9 runs.

In [ ]:
K = 20
tr20 = NextWordDataset(train_ids, K)
va20 = NextWordDataset(val_ids,   K)
te20 = NextWordDataset(test_ids,  K)
print(f'k={K}: {len(tr20):,} train | {len(va20):,} val | {len(te20):,} test windows')

P3_EMB, P3_LR, P3_BS = 256, 1e-3, 512
P3_MAX_EPOCHS, P3_PATIENCE = 20, 2

EXPERIMENTS = [   # (cell, layers, hidden, dropout)
    ('rnn',  1, 256, 0.0),   # RNN baseline
    ('rnn',  2, 256, 0.2),   # RNN: deeper
    ('rnn',  1, 512, 0.2),   # RNN: wider
    ('lstm', 1, 128, 0.0),   # LSTM: width 128
    ('lstm', 1, 256, 0.0),   # LSTM baseline
    ('lstm', 1, 256, 0.2),   # LSTM: + dropout
    ('lstm', 1, 512, 0.5),   # LSTM: width 512, strong dropout
    ('lstm', 2, 256, 0.2),   # LSTM: 2 layers
    ('lstm', 2, 512, 0.2),   # LSTM: 2 layers, wide
    ('lstm', 2, 512, 0.5),   # LSTM: 2 layers, wide, strong dropout
    ('lstm', 3, 512, 0.5),   # LSTM: 3 layers
]
if QUICK_RUN:
    EXPERIMENTS, P3_MAX_EPOCHS = [EXPERIMENTS[i] for i in (0, 1, 3, 7)], 2
run_name = lambda c, L, H, D: f'{c}_L{L}_H{H}_D{D}'
RUN_NAMES = [run_name(*e) for e in EXPERIMENTS]

def ckpt_file(entry):
    """Checkpoint of a Phase 3 run. Resolved inside CKPT, so it also works if the results
    folder was copied from Colab (where the stored path was a /content/drive/... path)."""
    return os.path.join(CKPT, os.path.basename(entry['ckpt']))

**Cell 3.2 – Running all experiments.** Each run starts from the same seed, trains for at most 20 epochs with early stopping on validation perplexity (patience 2, learning rate halved on the first plateau), and keeps the best-validation checkpoint. Test perplexity is computed for every run because the assignment asks for it in the table, but it is never used to choose a model. Results are saved after every run, and finished runs are skipped on a re-run if their checkpoint still exists and the data fingerprint matches.

In [ ]:
p3_path = f'{RES}/phase3.json'
p3 = json.load(open(p3_path)) if (os.path.exists(p3_path) and CACHE_VALID) else {}

for cell, L, H, D in EXPERIMENTS:
    name = run_name(cell, L, H, D)
    if name in p3 and os.path.exists(ckpt_file(p3[name])):
        print(f'{name}: done (val ppl {p3[name]["val_ppl"]:.2f}), skipping'); continue
    set_seed(SEED)
    model = NextWordModel(V, cell, P3_EMB, H, L, D).to(device)
    ckpt  = f'{CKPT}/p3_{name}.pt'
    print(f'=== {name}: {count_params(model):,} params')
    t0 = time.time()
    hist, best = fit(model, tr20, va20, epochs=P3_MAX_EPOCHS, lr=P3_LR, batch_size=P3_BS,
                     patience=P3_PATIENCE, ckpt=ckpt)
    test_ppl, test_acc = evaluate(model, te20)      # reported only, never used for selection
    best_ep = min(hist, key=lambda h: h['val_ppl'])
    p3[name] = dict(cell=cell, layers=L, hidden=H, dropout=D, emb_dim=P3_EMB,
                    params=count_params(model), epochs_run=len(hist), best_epoch=best_ep['epoch'],
                    val_ppl=best, val_acc=best_ep['val_acc'], test_ppl=test_ppl, test_acc=test_acc,
                    minutes=(time.time() - t0) / 60, ckpt=os.path.basename(ckpt), history=hist)
    json.dump(p3, open(p3_path, 'w'), indent=1)
    print(f'  -> best val ppl {best:.2f} (epoch {best_ep["epoch"]}) | test ppl {test_ppl:.2f} '
          f'| {p3[name]["minutes"]:.1f} min')
    del model
    if device.type == 'cuda': torch.cuda.empty_cache()

**Cell 3.3 – Results table, model selection, and saving the two best models.** The table lists every run in the experiment list, sorted by validation perplexity, with parameters, epochs, validation and test perplexity and accuracy, and training time. The best RNN and the best LSTM are the runs with the lowest **validation** perplexity, as the assignment requires.

Both are saved as stand-alone files, `checkpoints/best_rnn.pt` and `checkpoints/best_lstm.pt`. Each holds the weights and the architecture settings, so Phases 4–6 always rebuild and reload exactly these two models from disk and never depend on whatever model happens to be in memory. `best_models.json` keeps the summary for the report.

In [ ]:
p3_runs = {n: p3[n] for n in RUN_NAMES if n in p3}
df3 = pd.DataFrame([{'run': n, 'cell': r['cell'].upper(), 'layers': r['layers'], 'hidden': r['hidden'],
                     'dropout': r['dropout'], 'params (M)': round(r['params'] / 1e6, 2),
                     'epochs': r['epochs_run'], 'best ep': r['best_epoch'],
                     'val ppl': round(r['val_ppl'], 2), 'val acc %': round(100 * r['val_acc'], 2),
                     'test ppl': round(r['test_ppl'], 2), 'test acc %': round(100 * r['test_acc'], 2),
                     'min': round(r['minutes'], 1)} for n, r in p3_runs.items()])
df3 = df3.sort_values('val ppl').reset_index(drop=True)
display(df3)
df3.to_csv(f'{RES}/phase3_table.csv', index=False)

best = {}
for cell in ['rnn', 'lstm']:
    name = min((n for n in p3_runs if p3_runs[n]['cell'] == cell), key=lambda n: p3_runs[n]['val_ppl'])
    best[cell] = {k: v for k, v in p3_runs[name].items() if k != 'history'} | {'name': name}
    best[cell]['ckpt'] = os.path.basename(best[cell]['ckpt'])
    print(f'Best {cell.upper()}: {name} | val ppl {best[cell]["val_ppl"]:.2f} | test ppl {best[cell]["test_ppl"]:.2f}')
    e = best[cell]
    torch.save({'config': dict(cell=e['cell'], emb_dim=e['emb_dim'], hidden=e['hidden'],
                               layers=e['layers'], dropout=e['dropout']),
                'state_dict': torch.load(ckpt_file(e), map_location='cpu'),
                'name': name, 'val_ppl': e['val_ppl'], 'test_ppl': e['test_ppl'],
                'vocab_size': V, 'data_fingerprint': DATA_FP},
               f'{CKPT}/best_{cell}.pt')
    print(f'   saved -> {CKPT}/best_{cell}.pt')
json.dump(best, open(f'{RES}/best_models.json', 'w'), indent=1)

**Cell 3.4 – Learning curves.** Validation perplexity per epoch for every run, with RNNs dashed and LSTMs solid. The figure shows where each run peaked and began to overfit, how dropout delays that point, and how the RNNs plateau higher.

In [ ]:
plt.figure(figsize=(8, 5))
for n, r in p3_runs.items():
    ep = [h['epoch'] for h in r['history']]; vp = [h['val_ppl'] for h in r['history']]
    plt.plot(ep, vp, '--' if r['cell'] == 'rnn' else '-', marker='.', label=n)
plt.xlabel('epoch'); plt.ylabel('validation perplexity'); plt.title('Phase 3 learning curves (k = 20)')
plt.grid(alpha=.3); plt.legend(fontsize=7, ncol=2); plt.tight_layout()
plt.savefig(f'{RES}/phase3_curves.png', dpi=200); plt.show()

## Reloading the two best models (used by Phases 4, 5 and 6)

**Cell R.1 – Load `best_rnn.pt` and `best_lstm.pt` from disk.** Each model is rebuilt from the architecture stored in its file and its weights are loaded. The cell checks that the checkpoint was trained on the current vocabulary (same size and same data fingerprint). It also checks that it is still the run with the lowest validation perplexity in `best_models.json`, so a stale file can never be used by mistake. `lstm` is the name the Phase 4 chatbot uses.

In [ ]:
def load_best(cell):
    ck = torch.load(f'{CKPT}/best_{cell}.pt', map_location=device)
    cfg = ck['config']
    m = NextWordModel(V, cfg['cell'], cfg['emb_dim'], cfg['hidden'], cfg['layers'], cfg['dropout']).to(device)
    m.load_state_dict(ck['state_dict'])
    assert ck['vocab_size'] == V and ck['data_fingerprint'] == DATA_FP, f'best_{cell}.pt was trained on other data'
    return m.eval(), ck

best = json.load(open(f'{RES}/best_models.json'))
best_rnn,  info_rnn  = load_best('rnn')
best_lstm, info_lstm = load_best('lstm')
for cell, info in [('rnn', info_rnn), ('lstm', info_lstm)]:
    assert info['name'] == best[cell]['name'], f'best_{cell}.pt does not match best_models.json'
    print(f'Loaded best {cell.upper():4s}: {info["name"]:<18s} | val ppl {info["val_ppl"]:.2f} '
          f'| test ppl {info["test_ppl"]:.2f} | {count_params(best_rnn if cell == "rnn" else best_lstm):,} params')
lstm = best_lstm

## Phase 4 – Next-Word Prediction Chatbot

**Cell 4.1 – Generation.** `generate` works as follows. The seed phrase is tokenized with the same `tokenize` function as Phase 1, so it is lowercased with punctuation dropped. `<s>` is prepended, the words are mapped to ids, and unknown words become `<UNK>` and are reported back to the user. Each step feeds the model the last k = 20 tokens. A short seed is fine as it is, because an RNN can read a sequence of any length, so no padding is needed. The model is blocked from generating `<pad>`, `<s>` and `<UNK>`. When it produces `</s>`, a full stop is printed and `</s> <s>` is appended to the context, matching what the model saw in training. Only real words count toward N.

Greedy decoding picks the most probable word every time. It tends to fall into repetitive loops such as “i am sure i am sure”, which is worth mentioning in the report. Temperature sampling divides the scores by T before the softmax: T < 1 makes the output safer and more repetitive, T > 1 more varied and less coherent. `pretty` capitalises “I” and the start of each sentence for display.

In [ ]:
BANNED = torch.tensor([PAD_ID, UNK_ID, BOS_ID], device=device)

@torch.no_grad()
def generate(model, seed_text, n_words, mode='greedy', temperature=1.0, k=K):
    words   = tokenize(seed_text)
    unknown = [w for w in words if w not in stoi]
    ctx     = [BOS_ID] + [stoi.get(w, UNK_ID) for w in words]
    out, produced = [], 0
    for _ in range(4 * n_words):                       # safety cap in case of repeated </s>
        if produced >= n_words: break
        logits = model(torch.tensor([ctx[-k:]], device=device))[0].float()
        logits[BANNED] = -float('inf')                 # never emit <pad>, <s>, <UNK>
        if mode == 'greedy':
            nxt = int(logits.argmax())
        else:
            probs = torch.softmax(logits / temperature, dim=-1)
            nxt = int(torch.multinomial(probs.cpu(), 1))
        if nxt == EOS_ID:                              # sentence ended: start a new one
            out.append('.'); ctx += [EOS_ID, BOS_ID]
        else:
            out.append(itos[nxt]); ctx.append(nxt); produced += 1
    return out, unknown

def pretty(tokens, cap_first=False):
    words, cap = [], cap_first
    for t in tokens:
        if t == '.':
            if words: words[-1] += '.'
            cap = True; continue
        w = 'I' + t[1:] if (t == 'i' or t.startswith("i'")) else t
        if cap: w, cap = w[0].upper() + w[1:], False
        words.append(w)
    return ' '.join(words)

**Cell 4.2 – Fixed examples.** Four seed phrases (two Austen-like, two Shakespeare-like) are continued for 25 words with greedy decoding and with temperature sampling at T = 0.5, 1.0 and 1.5. The sampling seed is fixed, so these examples are reproducible and can be quoted in the report.

In [ ]:
seeds = ['it is a truth universally acknowledged', 'my dear lizzy', 'what light through yonder', 'the king shall']
torch.manual_seed(SEED)
for s in seeds:
    print(f'\nSEED: "{s}"')
    g, _ = generate(lstm, s, 25, 'greedy');        print(f'  greedy : {pretty(g)}')
    for T in [0.5, 1.0, 1.5]:
        g, _ = generate(lstm, s, 25, 'sample', T); print(f'  T={T:<4} : {pretty(g)}')

**Cell 4.3 – The interactive `input()` loop.** The loop asks for a seed phrase, then N, then the decoding method, then a temperature if sampling was chosen. Typing `quit` at any prompt exits. Each invalid case gets its own message and the loop continues instead of crashing: an empty seed or punctuation only, an N that is not a whole number or falls outside 1–200, an unknown decoding option, and a temperature that is not a number or falls outside 0.1–2.0. A general `except` around each turn catches anything unexpected, and stopping the cell (KeyboardInterrupt) closes the loop cleanly.

When the notebook runs without an input box (`jupyter nbconvert --execute`, or a headless *Run all*), `input()` raises `StdinNotImplementedError`. The loop recognises this and closes instead of printing an error forever. Set `RUN_CHATBOT = False` (Cell 0.1, or `A2_RUN_CHATBOT=0`) to skip the chatbot entirely.

In [ ]:
def chatbot(model, max_n=200):
    print("Next-word chatbot (best LSTM). Type 'quit' at any prompt to exit.")
    while True:
        try:
            seed = input('\nSeed phrase: ').strip()
            if seed.lower() == 'quit': break
            if not tokenize(seed):
                print('  Please type at least one word (letters).'); continue

            n_raw = input(f'Number of words to generate (1-{max_n}): ').strip()
            if n_raw.lower() == 'quit': break
            if not n_raw.isdigit() or not 1 <= int(n_raw) <= max_n:
                print(f'  N must be a whole number between 1 and {max_n}.'); continue
            n = int(n_raw)

            mode = input('Decoding - [g]reedy or [t]emperature sampling? ').strip().lower()
            if mode == 'quit': break
            if mode in ('g', 'greedy'):
                mode, temp = 'greedy', 1.0
            elif mode in ('t', 'temp', 'temperature'):
                t_raw = input('Temperature 0.1-2.0 (Enter = 0.8): ').strip()
                if t_raw.lower() == 'quit': break
                try:
                    temp = float(t_raw) if t_raw else 0.8
                except ValueError:
                    print('  Temperature must be a number like 0.7.'); continue
                if not 0.1 <= temp <= 2.0:
                    print('  Temperature must be between 0.1 and 2.0.'); continue
                mode = 'sample'
            else:
                print("  Please type 'g' or 't'."); continue

            gen, unknown = generate(model, seed, n, mode, temp)
            if unknown:
                print(f'  (not in vocabulary, read as <UNK>: {", ".join(unknown)})')
            label = 'greedy' if mode == 'greedy' else f'T={temp}'
            print(f'  [{label}] {seed} ... {pretty(gen)}')
        except (KeyboardInterrupt, EOFError):
            break
        except Exception as e:                       # anything unexpected: report and keep going
            if type(e).__name__ == 'StdinNotImplementedError':   # kernel has no stdin -> stop, don't loop forever
                print('  This kernel cannot read input(); skipping the chatbot.'); break
            print(f'  Something went wrong ({e}). Please try again.')
    print('Chatbot closed. Goodbye!')

if RUN_CHATBOT:
    chatbot(lstm)
else:
    print('RUN_CHATBOT is False - chatbot skipped.')

## Phase 5 – Perplexity and Confidence Analysis

**Cell 5.0 – Setup.** Imports for the statistics and regression, the constants for Phases 5–6 (k = 20, rare-word threshold 50) and an output folder, `BASE/outputs`, where every table and figure from Phases 5–6 is saved for the report. The device, seed and models come from the earlier cells.

`CROSS_SENTENCE_CONTEXT = True` matches how the models were trained in Phase 1: windows slide over the continuous token stream, so the 20-token context of a word early in a sentence includes the end of the previous sentence.

In [ ]:
import gc, copy, contextlib
from matplotlib.patches import Patch
from scipy.stats import wilcoxon, spearmanr
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error

set_seed(SEED)
assert K == 20, 'Phases 5-6 use k = 20'
RARE_THRESHOLD = 50             # a word is rare if it occurs < 50 times in the train split (<UNK> counts as rare)
CROSS_SENTENCE_CONTEXT = True   # True = context may reach into previous sentences, exactly as in the Phase 1 windows
EVAL_BS = 2048 if device.type == 'cuda' else 256
RESULTS = {}

OUT_DIR = f'{BASE}/outputs'
FIG_DIR = os.path.join(OUT_DIR, 'figures')
os.makedirs(FIG_DIR, exist_ok=True)

def savefig(name):
    plt.savefig(os.path.join(FIG_DIR, name), dpi=150, bbox_inches='tight')
    print('  saved figure ->', os.path.join(FIG_DIR, name))

print(f'outputs -> {OUT_DIR}')

**Cell 5.1 – Connecting to Phases 1–4.** Uses the objects from the earlier phases directly: `best_rnn` and `best_lstm` (reloaded from disk in Cell R.1), the vocabulary `stoi`/`itos` with `PAD_ID`, `UNK_ID`, `BOS_ID`, `EOS_ID`, the test and train sentences `test_sents`/`train_sents` (lists of words without boundary tokens, Cell 1.5), and `train_counts`, the word counts of the training split (Cell 1.6), for the rare-word feature. It then checks that each model maps a (batch, 20) tensor of word ids to (batch, |V|) next-word scores.

In [ ]:
MODELS = {'RNN': best_rnn, 'LSTM': best_lstm}
test_tok, train_tok = test_sents, train_sents        # list[list[str]], no <s>/</s>

def is_unk(w):  return w not in stoi
def is_rare(w): return is_unk(w) or train_counts.get(w, 0) < RARE_THRESHOLD

def last_logits(model, x):
    """(B, K) word ids -> (B, V) next-word scores."""
    return model(x)

with torch.no_grad():
    for name, m in MODELS.items():
        m.to(device).eval()
        o = last_logits(m, torch.full((2, K), BOS_ID, dtype=torch.long, device=device))
        assert tuple(o.shape) == (2, V), f'{name}: output {tuple(o.shape)}, expected (2, {V})'
        info = info_rnn if name == 'RNN' else info_lstm
        print(f'  {name}: {info["name"]} | {count_params(m)/1e6:.2f}M parameters - output shape OK')

L = np.array([len(s) for s in test_tok])
print(f'\nvocab |V| = {V:,} | ids: <pad>={PAD_ID}, <UNK>={UNK_ID}, <s>={BOS_ID}, </s>={EOS_ID}')
print(f'test sentences: {len(test_tok):,} (mean {L.mean():.1f} words, max {L.max()}) | train sentences: {len(train_tok):,}')

**Cell 5.2 – Scoring engine.** Each split is laid out as one token stream, `<s> w1 … wN </s> <s> …`, exactly as in Phase 1. Every word and every `</s>` is a prediction target; `<s>` never is, as the assignment specifies. For each target the model sees the previous k = 20 tokens. Contexts that would run past the start of the text are filled with `<pad>`, whose embedding is fixed at zero; this only affects the first 20 tokens of a split. With `CROSS_SENTENCE_CONTEXT=False`, anything before the sentence's own `<s>` would also be padded. Scoring runs in large batches in full float32 precision and records, for every target, log P(correct word), the model's top probability (its confidence), and whether the correct word was the top-1 / in the top-5.

In [ ]:
OFFS = np.arange(-K, 0)

def encode_words(words):
    return [stoi.get(w, UNK_ID) for w in words]

def build_stream(sents):
    """sents: list of word lists. Returns token stream + target positions (words and </s>),
    the sentence each target belongs to, and the stream index of that sentence's <s>."""
    stream, pos, owner, start = [], [], [], []
    for i, s in enumerate(sents):
        s0 = len(stream); stream.append(BOS_ID)
        for t in encode_words(s) + [EOS_ID]:
            pos.append(len(stream)); owner.append(i); start.append(s0); stream.append(t)
    return dict(stream=np.asarray(stream, np.int64), pos=np.asarray(pos, np.int64),
                owner=np.asarray(owner, np.int64), start=np.asarray(start, np.int64), n_sent=len(sents))

def contexts(S, sel):
    """k-token contexts and targets for the targets selected by `sel` (slice or index array)."""
    p = S['pos'][sel]
    idx = p[:, None] + OFFS[None, :]
    ctx = S['stream'][np.clip(idx, 0, None)]
    limit = 0 if CROSS_SENTENCE_CONTEXT else S['start'][sel][:, None]
    ctx[idx < limit] = PAD_ID
    return ctx, S['stream'][p]

@torch.no_grad()
def score(model, S, bs=EVAL_BS):
    model.eval()
    n = len(S['pos'])
    out = dict(logp=np.empty(n, np.float64), conf=np.empty(n, np.float32),
               top1=np.empty(n, bool), top5=np.empty(n, bool))
    for b in range(0, n, bs):
        sl = slice(b, min(b + bs, n))
        x, y = contexts(S, sl)
        x, y = torch.from_numpy(x).to(device), torch.from_numpy(y).to(device)
        lp = torch.log_softmax(last_logits(model, x).float(), dim=-1)
        top = lp.topk(5, dim=-1)
        out['logp'][sl] = lp.gather(1, y[:, None]).squeeze(1).double().cpu().numpy()
        out['conf'][sl] = top.values[:, 0].exp().cpu().numpy()
        out['top1'][sl] = (top.indices[:, 0] == y).cpu().numpy()
        out['top5'][sl] = (top.indices == y[:, None]).any(1).cpu().numpy()
    return out

def corpus_ppl(r, mask=None):
    lp = r['logp'] if mask is None else r['logp'][mask]
    return float(np.exp(-lp.mean()))

def summarize(r, S):
    y = S['stream'][S['pos']]; known = y != UNK_ID; nll = -r['logp']
    return dict(ppl=float(np.exp(nll.mean())), ppl_excl_unk=float(np.exp(nll[known].mean())),
                bits_per_token=float(nll.mean() / math.log(2)), top1=float(r['top1'].mean()),
                top5=float(r['top5'].mean()), mean_conf=float(r['conf'].mean()),
                unk_target_rate=float(1 - known.mean()), n_tokens=int(len(y)))
print('scoring helpers ready')

**Cell 5.3 – Scoring the test split.** Both models score every test token. The cell reports corpus perplexity, perplexity with `<UNK>` targets excluded, and top-1 / top-5 next-word accuracy; the accuracy numbers go in the report's Metrics section. **Sanity check:** the perplexity here should be very close to the Phase 3 test perplexity printed next to it. Small differences come from the first 20 test tokens (padded here, skipped in Phase 3) and from Phase 3 evaluating in mixed precision.

In [ ]:
test_S = build_stream(test_tok)
test_words = [w for s in test_tok for w in s]
print(f'test: {len(test_tok):,} sentences | {len(test_S["pos"]):,} predicted tokens (words + </s>) | '
      f'word <UNK> rate {np.mean([is_unk(w) for w in test_words]):.2%}\n')

res_te = {}
for m, model in MODELS.items():
    t0 = time.time()
    res_te[m] = score(model, test_S)
    s = summarize(res_te[m], test_S)
    RESULTS[f'phase5_test_{m}'] = s
    p3_test = (info_rnn if m == 'RNN' else info_lstm)['test_ppl']
    print(f'{m:4s}: test PPL {s["ppl"]:8.2f} (Phase 3: {p3_test:8.2f}) | PPL excl. <UNK> targets {s["ppl_excl_unk"]:8.2f} | '
          f'top-1 {s["top1"]:.2%} | top-5 {s["top5"]:.2%} | {time.time()-t0:.1f}s')

**Cell 5.4 – Per-sentence perplexity.** For each test sentence (from `<s>` to `</s>`), N is the number of words plus one for `</s>`. The cell sums −log P over those N targets and computes PP(W) = exp(−(1/N) Σ log P(wᵢ | context)), which equals P(w₁…w_N)^(−1/N), the formula in the assignment. It also records the features used later: length in words, share of rare words (training count < 50, `<UNK>` counted as rare), `<UNK>` share, mean model confidence and per-sentence top-1 accuracy.

In [ ]:
n_s = len(test_tok)
df = pd.DataFrame({'text': [' '.join(s) for s in test_tok], 'length': [len(s) for s in test_tok]})
df['n_pred']     = np.bincount(test_S['owner'], minlength=n_s)            # words + </s>
df['rare_share'] = [np.mean([is_rare(w) for w in s]) for s in test_tok]
df['unk_share']  = [np.mean([is_unk(w) for w in s]) for s in test_tok]
for m, r in res_te.items():
    own = test_S['owner']
    df[f'nll_{m}']    = np.bincount(own, weights=-r['logp'], minlength=n_s)
    df[f'logppl_{m}'] = df[f'nll_{m}'] / df['n_pred']
    df[f'ppl_{m}']    = np.exp(df[f'logppl_{m}'])
    df[f'conf_{m}']   = np.bincount(own, weights=r['conf'].astype(float), minlength=n_s) / df['n_pred']
    df[f'acc_{m}']    = np.bincount(own, weights=r['top1'].astype(float), minlength=n_s) / df['n_pred']

print(f'rare-word share over all test words: {np.mean([is_rare(w) for w in test_words]):.2%}')
display(df[['length', 'rare_share', 'ppl_RNN', 'ppl_LSTM', 'conf_LSTM', 'acc_LSTM']].describe().round(3))

**Cell 5.5 – Length groups.** Test sentences are sorted by length and cut into three equal-sized groups (short, medium, long). For each group the table gives the pooled perplexity (exp of total NLL over total tokens) plus the median and mean sentence perplexity. The mean is dominated by a few very hard short sentences, so the median is the robust summary. A paired Wilcoxon signed-rank test on per-sentence log-perplexity checks whether the LSTM's advantage over the RNN in each group is statistically significant. The left plot shows pooled perplexity per group; the right one shows the full distribution of sentence perplexities.

In [ ]:
GROUPS = ['short', 'medium', 'long']
order = np.argsort(df['length'].to_numpy(), kind='stable')
df['group'] = ''
for g, idx in zip(GROUPS, np.array_split(order, 3)):       # three equal-sized groups
    df.loc[idx, 'group'] = g

rows = []
for g in GROUPS:
    d = df[df['group'] == g]
    row = {'group': g, 'n_sent': len(d), 'len_min': int(d['length'].min()),
           'len_max': int(d['length'].max()), 'len_mean': d['length'].mean()}
    for m in MODELS:
        row[f'{m}_pooled_ppl'] = np.exp(d[f'nll_{m}'].sum() / d['n_pred'].sum())
        row[f'{m}_median_ppl'] = d[f'ppl_{m}'].median()
        row[f'{m}_mean_ppl']   = d[f'ppl_{m}'].mean()
    diff = d['logppl_RNN'] - d['logppl_LSTM']
    row['LSTM_better_%'] = 100 * (diff > 0).mean()
    row['wilcoxon_p'] = wilcoxon(diff).pvalue if (diff != 0).any() else np.nan
    rows.append(row)
grp_tab = pd.DataFrame(rows).set_index('group')
RESULTS['phase5_length_groups'] = grp_tab.to_dict()
display(grp_tab.round(3))

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
x, w = np.arange(3), 0.38
labels = [f'{g}\n{grp_tab.loc[g, "len_min"]}-{grp_tab.loc[g, "len_max"]} words' for g in GROUPS]
for i, m in enumerate(MODELS):
    bars = axes[0].bar(x + (i - 0.5) * w, grp_tab[f'{m}_pooled_ppl'], w, label=m)
    axes[0].bar_label(bars, fmt='%.0f', fontsize=8)
axes[0].set_xticks(x); axes[0].set_xticklabels(labels)
axes[0].set_ylabel('perplexity (pooled over tokens)'); axes[0].set_title('Test perplexity by sentence length'); axes[0].legend()

data, positions, colors = [], [], []
for j, g in enumerate(GROUPS):
    for i, m in enumerate(MODELS):
        data.append(df.loc[df['group'] == g, f'ppl_{m}'].values); positions.append(j + (i - 0.5) * w); colors.append(f'C{i}')
bp = axes[1].boxplot(data, positions=positions, widths=w * 0.9, showfliers=False, patch_artist=True)
for patch, c in zip(bp['boxes'], colors): patch.set_facecolor(c); patch.set_alpha(0.6)
axes[1].set_yscale('log'); axes[1].set_xticks(x); axes[1].set_xticklabels(labels)
axes[1].set_ylabel('sentence perplexity (log scale)'); axes[1].set_title('Distribution of sentence perplexity')
axes[1].legend(handles=[Patch(color=f'C{i}', alpha=0.6, label=m) for i, m in enumerate(MODELS)])
plt.tight_layout(); savefig('phase5_length_groups.png'); plt.show()

**Cell 5.6 – Can simple features predict difficulty?** A linear regression predicts sentence log-perplexity from two features: sentence length and rare-word share. It is fitted on 70% of the test sentences and evaluated on the other 30%, using the same seeded split for both models. R² and RMSE are reported, together with a baseline that always predicts the training mean, so R² has a reference point. RMSE is in nats of log-perplexity, so exp(RMSE) is the typical multiplicative error in perplexity. Standardized coefficients show the effect of a one-standard-deviation change in each feature, which tells you which feature matters more.

In [ ]:
FEATURES = ['length', 'rare_share']
X = df[FEATURES].to_numpy(float)
tr_idx, ho_idx = train_test_split(np.arange(len(df)), test_size=0.30, random_state=SEED)

reg_rows = []
fig, axes = plt.subplots(1, len(MODELS), figsize=(12, 4.5))
for ax, m in zip(np.atleast_1d(axes), MODELS):
    y = df[f'logppl_{m}'].to_numpy()
    reg = LinearRegression().fit(X[tr_idx], y[tr_idx])
    pred = reg.predict(X[ho_idx])
    rmse = math.sqrt(mean_squared_error(y[ho_idx], pred))
    base_rmse = math.sqrt(mean_squared_error(y[ho_idx], np.full(len(ho_idx), y[tr_idx].mean())))
    std_coef = reg.coef_ * X[tr_idx].std(0)
    reg_rows.append({'model': m, 'R2_heldout': r2_score(y[ho_idx], pred), 'RMSE_heldout': rmse,
                     'RMSE_mean_baseline': base_rmse, 'intercept': reg.intercept_,
                     'coef_length': reg.coef_[0], 'coef_rare_share': reg.coef_[1],
                     'std_coef_length': std_coef[0], 'std_coef_rare_share': std_coef[1],
                     'n_fit': len(tr_idx), 'n_eval': len(ho_idx)})
    print(f'{m}: log-PPL = {reg.intercept_:.3f} + {reg.coef_[0]:+.4f}*length + {reg.coef_[1]:+.3f}*rare_share | '
          f'held-out R2 = {r2_score(y[ho_idx], pred):.3f}, RMSE = {rmse:.3f} (baseline {base_rmse:.3f})')
    lims = [min(y[ho_idx].min(), pred.min()), max(y[ho_idx].max(), pred.max())]
    ax.scatter(y[ho_idx], pred, s=6, alpha=0.35, c=df['rare_share'].to_numpy()[ho_idx], cmap='viridis')
    ax.plot(lims, lims, 'k--', lw=1)
    ax.set_xlabel('actual sentence log-perplexity'); ax.set_ylabel('predicted')
    ax.set_title(f'{m}: held-out 30% (colour = rare share)')
reg_tab = pd.DataFrame(reg_rows).set_index('model')
RESULTS['phase5_regression'] = reg_tab.to_dict()
plt.tight_layout(); savefig('phase5_regression.png'); plt.show()
display(reg_tab.round(4))

**Cell 5.7 – Confidence analysis.** This asks whether the model knows when it is likely to be wrong. The reliability diagram bins every test prediction by the model's top probability and compares that confidence with the actual top-1 accuracy in the bin; a perfectly calibrated model lies on the diagonal. Expected Calibration Error (ECE) summarizes the gap in one number. At sentence level, Spearman correlations show whether mean confidence, length and rare share track sentence difficulty. The cell ends with the easiest and hardest test sentences (at least 5 words) so the patterns can be inspected directly.

In [ ]:
def reliability(conf, correct, n_bins=10):
    edges = np.linspace(0, 1, n_bins + 1)
    b = np.clip(np.digitize(conf, edges) - 1, 0, n_bins - 1)
    cnt = np.bincount(b, minlength=n_bins)
    acc = np.bincount(b, weights=correct.astype(float), minlength=n_bins) / np.maximum(cnt, 1)
    cf  = np.bincount(b, weights=conf.astype(float), minlength=n_bins) / np.maximum(cnt, 1)
    return edges, cnt, acc, cf, float(np.sum(cnt / cnt.sum() * np.abs(acc - cf)))

fig, axes = plt.subplots(1, len(MODELS), figsize=(11, 4.5))
for ax, m in zip(np.atleast_1d(axes), MODELS):
    edges, cnt, acc, cf, ece = reliability(res_te[m]['conf'], res_te[m]['top1'])
    ok = cnt > 0
    ax.bar(edges[:-1][ok], acc[ok], width=0.1, align='edge', alpha=0.7, edgecolor='k', label='accuracy')
    ax.plot([0, 1], [0, 1], 'k--', lw=1, label='perfect calibration')
    ax.plot(cf[ok], acc[ok], 'o-', color='C3', ms=4, label='(confidence, accuracy)')
    ax.set_xlabel('model confidence (top probability)'); ax.set_ylabel('top-1 accuracy')
    ax.set_title(f'{m}: ECE = {ece:.3f}'); ax.legend(fontsize=8)
    RESULTS[f'phase5_calibration_{m}'] = dict(ECE=ece, mean_conf=float(res_te[m]['conf'].mean()),
                                              top1=float(res_te[m]['top1'].mean()))
    print(f'{m}: mean confidence {res_te[m]["conf"].mean():.3f} vs top-1 accuracy {res_te[m]["top1"].mean():.3f} | ECE {ece:.3f}')
plt.tight_layout(); savefig('phase5_reliability.png'); plt.show()

print('\nSpearman correlation with sentence log-perplexity:')
for m in MODELS:
    for feat in ['length', 'rare_share', f'conf_{m}']:
        rho, p = spearmanr(df[feat], df[f'logppl_{m}'])
        print(f'  {m:4s} vs {feat:<11s}: rho = {rho:+.3f} (p = {p:.1e})')

cand = df[df['length'] >= 5]
for title, sub in [('EASIEST', cand.nsmallest(5, 'ppl_LSTM')), ('HARDEST', cand.nlargest(5, 'ppl_LSTM'))]:
    print(f'\n{title} test sentences for the LSTM (>= 5 words):')
    for _, r in sub.iterrows():
        print(f"  PPL {r['ppl_LSTM']:9.1f} | len {int(r['length']):3d} | rare {r['rare_share']:4.0%} | {r['text'][:100]}")

## Phase 6 – Multi-Style Adaptation (BBC News)

**Cell 6.1 – BBC articles without duplicates.** This starts from the articles Phase 1 already loaded (`bbc_articles`, Cell 1.9): one copy per (category, file name), title line removed, the same text normalisation. Phase 1 counted, but did not remove, articles whose body text is identical under a different file name. They are dropped here, because a repeat landing in both the adaptation and the evaluation split would leak test text into training.

In [ ]:
BBC_CATS = ['business', 'entertainment', 'politics', 'sport', 'tech']
bbc = pd.DataFrame(bbc_articles)
n_before = len(bbc)
bbc = bbc[~bbc.duplicated('md5', keep='first')].sort_values(['category', 'file']).reset_index(drop=True)
n_content_dup = n_before - len(bbc)

RESULTS['phase6_bbc_dedup'] = dict(after_removing_copies=n_before, exact_duplicates_dropped=n_content_dup,
                                   final_articles=len(bbc))
print(f'articles after removing the duplicate copy: {n_before:,} -> after dropping {n_content_dup} exact repeats: {len(bbc):,}')
display(bbc['category'].value_counts().to_frame('articles'))

**Cell 6.2 – Same preprocessing, frozen vocabulary.** Each article body goes through exactly the Phase 1 pipeline (paragraphs → `sent_tokenize` → `tokenize`). Words are mapped with the **training vocabulary from the primary corpus**, which is never rebuilt, so every word the model never saw becomes `<UNK>`. The cell compares the `<UNK>` rate and rare-word share of the in-domain test split with BBC, and lists the most frequent BBC words the model has never seen. Articles are then split 70/10/20 into adapt/val/eval, stratified by category and seeded, for the adaptation experiment in Cell 6.5.

In [ ]:
bbc['sents'] = bbc['body'].map(lambda b: to_sentences(bbc_body_paragraphs(b)))
bbc_words = [w for ss in bbc['sents'] for s in ss for w in s]
oov = {
    'domain': ['primary test', 'BBC News'],
    'sentences': [len(test_tok), int(bbc['sents'].map(len).sum())],
    'words': [len(test_words), len(bbc_words)],
    'UNK_rate_%': [100 * np.mean([is_unk(w) for w in test_words]), 100 * np.mean([is_unk(w) for w in bbc_words])],
    'rare_share_%': [100 * np.mean([is_rare(w) for w in test_words]), 100 * np.mean([is_rare(w) for w in bbc_words])],
}
oov_tab = pd.DataFrame(oov).set_index('domain')
RESULTS['phase6_oov'] = oov_tab.to_dict()
display(oov_tab.round(2))
print('Most frequent BBC words missing from the training vocabulary:')
print(Counter(w for w in bbc_words if is_unk(w)).most_common(30))

# 70% adapt / 10% val / 20% eval, by article, stratified by category
art_idx = np.arange(len(bbc))
bbc_adapt_idx, rest_idx = train_test_split(art_idx, test_size=0.30, random_state=SEED, stratify=bbc['category'])
bbc_val_idx, bbc_eval_idx = train_test_split(rest_idx, test_size=2/3, random_state=SEED,
                                             stratify=bbc['category'].iloc[rest_idx])
bbc['split'] = 'adapt'
bbc.loc[bbc_val_idx, 'split'] = 'val'
bbc.loc[bbc_eval_idx, 'split'] = 'eval'
display(bbc.groupby(['split', 'category']).size().unstack())

def bbc_subset(mask):
    sub = bbc[mask]
    sents = [s for ss in sub['sents'] for s in ss]
    cats = np.array([c for c, ss in zip(sub['category'], sub['sents']) for _ in ss])
    return sents, cats

**Cell 6.3 – The generalization gap.** Both best models, trained only on Austen and Shakespeare, score the full deduplicated BBC corpus. The gap is reported three ways: the absolute change in perplexity, the ratio of perplexities, and the extra bits per token (the cross-entropy difference, which is the most interpretable). With a frozen vocabulary many BBC words become `<UNK>`, and `<UNK>` is relatively easy to predict, so the raw perplexity understates the gap. Perplexity with `<UNK>` targets excluded is therefore reported too. A per-category breakdown shows which news style is furthest from 19th-century prose and Elizabethan drama.

In [ ]:
bbc_all_sents, bbc_all_cats = bbc_subset(np.ones(len(bbc), bool))
bbc_S = build_stream(bbc_all_sents)
res_bbc = {}
for m, model in MODELS.items():
    t0 = time.time(); res_bbc[m] = score(model, bbc_S)
    print(f'{m}: scored {len(bbc_S["pos"]):,} BBC tokens in {time.time()-t0:.1f}s')

rows = []
for m in MODELS:
    a, b = summarize(res_te[m], test_S), summarize(res_bbc[m], bbc_S)
    rows.append({'model': m, 'test_ppl': a['ppl'], 'bbc_ppl': b['ppl'],
                 'delta_ppl': b['ppl'] - a['ppl'], 'change_%': 100 * (b['ppl'] / a['ppl'] - 1),
                 'ratio': b['ppl'] / a['ppl'], 'delta_bits_per_token': b['bits_per_token'] - a['bits_per_token'],
                 'test_ppl_excl_unk': a['ppl_excl_unk'], 'bbc_ppl_excl_unk': b['ppl_excl_unk'],
                 'ratio_excl_unk': b['ppl_excl_unk'] / a['ppl_excl_unk'],
                 'test_top1': a['top1'], 'bbc_top1': b['top1'],
                 'test_unk_targets': a['unk_target_rate'], 'bbc_unk_targets': b['unk_target_rate']})
    RESULTS[f'phase6_bbc_{m}'] = b
gap_tab = pd.DataFrame(rows).set_index('model')
RESULTS['phase6_gap'] = gap_tab.to_dict()
display(gap_tab.T.round(3))

tok_cat = bbc_all_cats[bbc_S['owner']]
cat_rows = []
for c in BBC_CATS:
    msk = tok_cat == c
    row = {'category': c, 'tokens': int(msk.sum())}
    for m in MODELS: row[f'{m}_ppl'] = corpus_ppl(res_bbc[m], msk)
    cat_rows.append(row)
cat_tab = pd.DataFrame(cat_rows).set_index('category')
RESULTS['phase6_categories'] = cat_tab.to_dict()
display(cat_tab.round(2))

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
x, w = np.arange(len(MODELS)), 0.38
b1 = axes[0].bar(x - w/2, gap_tab['test_ppl'], w, label='primary test (in-domain)')
b2 = axes[0].bar(x + w/2, gap_tab['bbc_ppl'], w, label='BBC News (out-of-domain)')
axes[0].bar_label(b1, fmt='%.0f'); axes[0].bar_label(b2, fmt='%.0f')
axes[0].set_xticks(x); axes[0].set_xticklabels(list(MODELS)); axes[0].set_ylabel('perplexity')
axes[0].set_title('Generalization gap'); axes[0].legend()
xc = np.arange(len(BBC_CATS))
for i, m in enumerate(MODELS):
    axes[1].bar(xc + (i - 0.5) * w, cat_tab[f'{m}_ppl'], w, label=f'{m} on BBC', color=f'C{i}')
    axes[1].axhline(gap_tab.loc[m, 'test_ppl'], color=f'C{i}', ls='--', lw=1, label=f'{m} in-domain test')
axes[1].set_xticks(xc); axes[1].set_xticklabels(BBC_CATS); axes[1].set_ylabel('perplexity')
axes[1].set_title('BBC perplexity by news category'); axes[1].legend(fontsize=8)
plt.tight_layout(); savefig('phase6_generalization_gap.png'); plt.show()

**Cell 6.4 – Decomposing the gap by word frequency.** Every target word is put into a band by its training-set frequency: frequent (≥ 1000), mid (50–999), rare (< 50), `<UNK>`, or `</s>`. For each domain the cell shows what share of tokens falls in each band and the perplexity on that band. This separates two causes of the gap. One is a vocabulary shift: BBC simply uses more rare and unknown words. The other is a style/syntax shift: even common words become harder to predict because the contexts they appear in are different.

In [ ]:
cnt_by_id = np.array([train_counts.get(w, 0) for w in itos])
BANDS = ['frequent (>=1000)', 'mid (50-999)', 'rare (<50)', '<UNK>', '</s>']

def band_of(y):
    c = cnt_by_id[y]
    b = np.where(c >= 1000, BANDS[0], np.where(c >= RARE_THRESHOLD, BANDS[1], BANDS[2])).astype(object)
    b[y == UNK_ID] = BANDS[3]; b[y == EOS_ID] = BANDS[4]
    return b

rows = []
for dom, S, R in [('primary test', test_S, res_te), ('BBC', bbc_S, res_bbc)]:
    b = band_of(S['stream'][S['pos']])
    for band in BANDS:
        msk = b == band
        row = {'domain': dom, 'band': band, 'share_%': 100 * msk.mean()}
        for m in MODELS: row[f'{m}_ppl'] = corpus_ppl(R[m], msk) if msk.any() else np.nan
        rows.append(row)
band_tab = pd.DataFrame(rows)
RESULTS['phase6_bands'] = band_tab.to_dict(orient='records')
display(band_tab.pivot(index='band', columns='domain').reindex(BANDS).round(2))

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
xb, w = np.arange(len(BANDS)), 0.38
for i, dom in enumerate(['primary test', 'BBC']):
    t = band_tab[band_tab['domain'] == dom].set_index('band').reindex(BANDS)
    axes[0].bar(xb + (i - 0.5) * w, t['share_%'], w, label=dom)
    axes[1].bar(xb + (i - 0.5) * w, t['LSTM_ppl'], w, label=dom)
for ax, title in zip(axes, ['share of target tokens (%)', 'best-LSTM perplexity on the band (log)']):
    ax.set_xticks(xb); ax.set_xticklabels(BANDS, rotation=15); ax.set_title(title); ax.legend()
axes[1].set_yscale('log')
plt.tight_layout(); savefig('phase6_frequency_bands.png'); plt.show()

**Cell 6.5 – Specialized vs. generalized training data.** Copies of the best LSTM are fine-tuned in two ways: on BBC alone (**specialized**), and on the same BBC windows mixed with an equal number of primary-corpus training windows (**generalized**). Each copy is then evaluated on the held-out BBC eval articles and on the original primary test set. The vocabulary stays frozen in both runs. Training uses Adam (lr 2e-4), gradient clipping at 1.0 and mixed precision on the GPU. The best epoch is chosen by BBC-*validation* perplexity; the eval split is never used for selection. The expected pattern: the specialized model closes most of the BBC gap but forgets some Austen/Shakespeare, while the generalized model keeps both. Note that the generalized run does twice as many updates per epoch, because of the rehearsal data. The original `best_lstm` is never modified.

In [ ]:
FT_MODELS = ['LSTM']            # add 'RNN' to repeat the experiment for the best RNN
FT_EPOCHS, FT_LR, FT_BS = 3, 2e-4, 512
MAX_FT_WINDOWS = 600_000        # cap on BBC training windows (None = use all); keeps runtime to a few minutes
if QUICK_RUN:
    FT_EPOCHS, MAX_FT_WINDOWS = 1, 20_000

def make_xy(S, n=None, seed=SEED):
    sel = np.arange(len(S['pos']))
    if n is not None and n < len(sel):
        sel = np.sort(np.random.default_rng(seed).choice(len(sel), n, replace=False))
    return contexts(S, sel)

def finetune(base, X, y, val_S, tag):
    set_seed(SEED)
    model = copy.deepcopy(base).to(device)
    for mod in model.modules():
        if isinstance(mod, nn.RNNBase): mod.flatten_parameters()
    opt, scaler = torch.optim.Adam(model.parameters(), lr=FT_LR), make_scaler()
    Xt, yt = torch.from_numpy(X), torch.from_numpy(y)
    gen = torch.Generator().manual_seed(SEED)
    best_ppl = corpus_ppl(score(model, val_S))
    best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    hist = [best_ppl]
    print(f'  [{tag}] epoch 0 (no fine-tuning): BBC-val PPL {best_ppl:.2f}')
    for ep in range(1, FT_EPOCHS + 1):
        model.train(); t0 = time.time(); tot = torch.zeros((), device=device)
        perm = torch.randperm(len(yt), generator=gen)
        for b in range(0, len(perm), FT_BS):
            i = perm[b:b + FT_BS]
            xb, yb = Xt[i].to(device, non_blocking=True), yt[i].to(device, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=USE_AMP):
                logits = last_logits(model, xb)
            loss = F.cross_entropy(logits.float(), yb)
            scaler.scale(loss).backward()
            scaler.unscale_(opt); nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update()
            tot += loss.detach() * len(i)
        v = corpus_ppl(score(model, val_S)); hist.append(v)
        print(f'  [{tag}] epoch {ep}: train loss {tot.item()/len(yt):.3f} | BBC-val PPL {v:.2f} | {time.time()-t0:.0f}s')
        if v < best_ppl:
            best_ppl, best_state = v, {k: t.detach().cpu().clone() for k, t in model.state_dict().items()}
    model.load_state_dict(best_state); model.eval()
    return model, hist

bbc_adapt_S = build_stream(bbc_subset(bbc['split'] == 'adapt')[0])
bbc_val_S   = build_stream(bbc_subset(bbc['split'] == 'val')[0])
bbc_eval_S  = build_stream(bbc_subset(bbc['split'] == 'eval')[0])
X_b, y_b = make_xy(bbc_adapt_S, MAX_FT_WINDOWS)
print(f'BBC adapt windows: {len(y_b):,} | val tokens: {len(bbc_val_S["pos"]):,} | eval tokens: {len(bbc_eval_S["pos"]):,}')
X_p, y_p = make_xy(build_stream(train_tok), len(y_b), seed=SEED + 1)
X_mix, y_mix = np.concatenate([X_b, X_p]), np.concatenate([y_b, y_p])
print(f'generalised set: {len(y_b):,} BBC + {len(y_p):,} primary-train windows')

ft_rows, ft_hist = [], {}
for m in FT_MODELS:
    base = MODELS[m]
    variants = {'base (no adaptation)': base}
    print(f'\n=== {m}: specialised (BBC only) ===')
    variants['specialised (BBC only)'], ft_hist[f'{m} specialised'] = finetune(base, X_b, y_b, bbc_val_S, f'{m}-spec')
    print(f'=== {m}: generalised (BBC + primary) ===')
    variants['generalised (BBC + primary)'], ft_hist[f'{m} generalised'] = finetune(base, X_mix, y_mix, bbc_val_S, f'{m}-gen')
    base_b = base_p = None
    for vname, vm in variants.items():
        pb, pp = corpus_ppl(score(vm, bbc_eval_S)), corpus_ppl(score(vm, test_S))
        if base_b is None: base_b, base_p = pb, pp
        ft_rows.append({'model': m, 'variant': vname, 'BBC_eval_ppl': pb, 'primary_test_ppl': pp,
                        'BBC_change_%': 100 * (pb / base_b - 1), 'primary_change_%': 100 * (pp / base_p - 1)})
    del variants; gc.collect()
    if device.type == 'cuda': torch.cuda.empty_cache()

ft_tab = pd.DataFrame(ft_rows)
RESULTS['phase6_adaptation'] = ft_tab.to_dict(orient='records')
RESULTS['phase6_adaptation_val_curves'] = ft_hist
display(ft_tab.round(2))

**Cell 6.6 – Visualizing adaptation.** The left panel shows each variant's perplexity on the out-of-domain eval split and on the in-domain test split; this is the specialization/forgetting trade-off. The right panel shows BBC-validation perplexity per fine-tuning epoch (epoch 0 is the unadapted model).

**Points to address in the report discussion:** (1) the size of the gap in bits per token, and how much of it is vocabulary (`<UNK>`/rare share, Cell 6.4) versus style (higher perplexity even on frequent words); (2) which BBC category is hardest and a plausible reason (e.g. the formulaic vocabulary of sports or business reporting vs. the narrative language of entertainment); (3) whether the LSTM's advantage over the RNN survives out of domain; (4) what the specialized vs. generalized runs say about narrow vs. broad training data, including the catastrophic forgetting visible in `primary_change_%`.

In [ ]:
for m in FT_MODELS:
    t = ft_tab[ft_tab['model'] == m].reset_index(drop=True)
    fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
    x, w = np.arange(len(t)), 0.38
    b1 = axes[0].bar(x - w/2, t['BBC_eval_ppl'], w, label='BBC eval (out-of-domain)')
    b2 = axes[0].bar(x + w/2, t['primary_test_ppl'], w, label='primary test (in-domain)')
    axes[0].bar_label(b1, fmt='%.0f'); axes[0].bar_label(b2, fmt='%.0f')
    axes[0].set_xticks(x); axes[0].set_xticklabels(t['variant'], rotation=10)
    axes[0].set_ylabel('perplexity'); axes[0].set_title(f'{m}: specialised vs generalised adaptation'); axes[0].legend()
    for key, h in ft_hist.items():
        if key.startswith(m):
            axes[1].plot(range(len(h)), h, marker='o', label=key)
    axes[1].set_xlabel('fine-tuning epoch (0 = original model)'); axes[1].set_ylabel('BBC-val perplexity')
    axes[1].set_title('Adaptation curves'); axes[1].legend()
    plt.tight_layout(); savefig(f'phase6_adaptation_{m}.png'); plt.show()

## Phase 7 – Results export and submission packaging

**Cell 7.1 – Exporting results.** Saves every table from Phases 5–6 as CSV and all numeric results as one JSON file in `BASE/outputs/`. Figures are already in `BASE/outputs/figures/`; the Phase 1–3 tables and figures are in `BASE/results/`. The report can be written straight from these files, and a grader can check the numbers without re-running.

In [ ]:
def _jsonable(o):
    if isinstance(o, (np.integer, np.floating, np.bool_)): return o.item()
    if isinstance(o, np.ndarray): return o.tolist()
    return str(o)

summary.to_csv(os.path.join(OUT_DIR, 'phase1_summary.csv'), index=False)
for name in ['df', 'grp_tab', 'reg_tab', 'gap_tab', 'cat_tab', 'band_tab', 'oov_tab', 'ft_tab']:
    obj = globals().get(name)
    if isinstance(obj, pd.DataFrame):
        path = os.path.join(OUT_DIR, f'{name}.csv'); obj.to_csv(path); print('saved', path)

RESULTS['best_models'] = {c: {k: v for k, v in e.items()} for c, e in best.items()}
RESULTS['data_fingerprint'] = DATA_FP
with open(os.path.join(OUT_DIR, 'results_phase5_6.json'), 'w') as f:
    json.dump(RESULTS, f, indent=2, default=_jsonable)
print('saved', os.path.join(OUT_DIR, 'results_phase5_6.json'))
print('Phase 5-6 figures:', sorted(os.listdir(FIG_DIR)))
print('Phase 1-3 results:', sorted(os.listdir(RES)))

if ON_COLAB and not BASE.startswith('/content/drive'):
    print(f"\nColab files disappear when the runtime ends - download '{BASE}' or copy it to Google Drive.")

**Cell 7.2 – README.txt.** Writes the run instructions that must go in the submission ZIP, next to the notebook. Fill in your name and roll number before submitting.

In [ ]:
README = r"""
Name: <MY NAME>            Roll No: <MY ROLL NO>
CS4045 Deep Learning for Perception - Assignment 2: RNNs vs LSTMs for Next-Word Prediction

CONTENTS
  notebooks/A2.ipynb   all experiments (Phases 1-7) in one notebook, fixed random seed (42)
  report.pdf           formal report
  README.txt           this file

DATA (Kaggle)
  Jane Austen      : kaggle.com/datasets/joshmcadams/jane-austin-and-charles-dickens
  Tiny Shakespeare : kaggle.com/datasets/thedevastator/the-bards-best-a-character-modeling-dataset
  BBC News         : kaggle.com/datasets/shivamkushwaha/bbc-full-text-document-classification
  The folders are found automatically (BASE/data/<name>, /kaggle/input, /content, ./data, ~/data);
  if they are not found, the notebook downloads them with kagglehub. To point at a folder by hand,
  set AUSTEN_DIR / SHAKESPEARE_DIR / BBC_DIR in Cell 1.1 or as environment variables.

OUTPUT (all under one working folder BASE)
  Colab: /content/drive/MyDrive/DL_A2   Kaggle: /kaggle/working/DL_A2   elsewhere: ./DL_A2
  BASE/processed/phase1.pkl         Phase 1 data
  BASE/checkpoints/                 Phase 2/3 checkpoints, best_rnn.pt and best_lstm.pt
  BASE/results/                     Phase 2/3 JSON, tables and figures
  BASE/outputs/                     Phase 5/6 CSV tables, results_phase5_6.json, figures/

RUN ON GOOGLE COLAB / KAGGLE
  1. Runtime -> Change runtime type -> GPU (Kaggle: Accelerator -> GPU, Internet on).
  2. Upload the notebook. Run all cells top to bottom (Colab asks to mount Google Drive).
  3. The Phase 4 chatbot waits for input: type 'quit' to continue
     (or set RUN_CHATBOT = False in Cell 0.1 before 'Run all').

RUN ON A LOCAL / UNIVERSITY GPU (e.g. RTX 3080, 10 GB)
  python -m venv .venv && source .venv/bin/activate
  pip install torch      (CUDA build; check: python -c "import torch; print(torch.cuda.is_available())")
  pip install numpy pandas matplotlib scikit-learn scipy nltk kagglehub jupyter nbconvert ipykernel
  python -m nltk.downloader punkt punkt_tab
  Non-interactive run (the chatbot closes itself when no input is possible):
  jupyter nbconvert --to notebook --execute A2.ipynb --output A2_executed.ipynb --ExecutePreprocessor.timeout=-1
  Interactive: jupyter lab --no-browser --port 8888, then on your laptop
  ssh -L 8888:localhost:8888 <user>@<gpu-host> and open http://localhost:8888

OPTIONAL ENVIRONMENT VARIABLES
  A2_RUN_CHATBOT=0     skip the chatbot          A2_FORCE_RETRAIN=1   ignore cached Phase 2/3 results
  A2_BASE=/path        working folder            A2_QUICK_RUN=1       2-5 minute smoke test on a data slice
                                                                      (separate folder; not for the report)

NOTES
  The device is selected automatically (CUDA if available, otherwise CPU).
  Phases 2 and 3 skip runs that are already saved (and made from the same data); a full run from
  scratch takes roughly 1-1.5 h on an RTX 3080 and 2-3x longer on a Colab T4.
  All randomness is seeded (SEED = 42); results can differ slightly between GPU models (cuDNN, fp16).
"""
with open('README.txt', 'w') as f:
    f.write(README.strip() + '\n')
print('written:', os.path.abspath('README.txt'))
print(open('README.txt').read())